#Importing Packages

In [ ]:
import torch
import torchvision
from torch.utils.data import DataLoader,Dataset,random_split,TensorDataset
import torch.optim as optim
import torch.nn as nn
import os
import copy
import sentencepiece as spm
import torch.nn.functional as F

#Collecting Data

In [ ]:
!unzip -q fra-eng.zip

In [ ]:
file_path = "/content/fra.txt"

In [ ]:
with open(file_path, 'r' , encoding='utf-8') as f:
  lines = f.readlines()
print(len(lines))

240521


#Cleaning

In [ ]:
pairs = []
for line in lines:
  parts = line.strip().split('\t')
  if len(parts) >= 2:
    eng = parts[0]
    fr = parts[1]
    pairs.append((eng,fr))
print(len(pairs))
print(pairs[:5])

240521
[('Go.', 'Va !'), ('Go.', 'Marche.'), ('Go.', 'En route !'), ('Go.', 'Bouge !'), ('Hi.', 'Salut !')]


In [ ]:
import re
def clean_text(s):
  s = s.lower().strip()
  s = re.sub(r"[^a-zA-Z?.!,]+"," ",s)
  s = re.sub(r"\s+", " ", s)
  return s
cleaned_pairs = [(clean_text(e),clean_text(f)) for e,f in pairs]
print(cleaned_pairs[:5])

[('go.', 'va !'), ('go.', 'marche.'), ('go.', 'en route !'), ('go.', 'bouge !'), ('hi.', 'salut !')]


In [ ]:
maxLen = 25
filtered_pairs = []
for e,f in cleaned_pairs:
  if len(e.split()) <= maxLen and len(f.split()) <= maxLen:
    filtered_pairs.append((e,f))
filtered_pairs = filtered_pairs[:40000]
print(len(filtered_pairs))


40000


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


#Padding and tokenization

In [ ]:
train_size = int(0.8*len(filtered_pairs))
val_size = len(filtered_pairs) - train_size
train_pairs, val_pairs = random_split(filtered_pairs, [train_size,val_size])
train_pairs[0]

('are you blushing?', 'rougissez vous ?')

In [ ]:
def add_tokens(pairs):
  return [("<sos> "+e+" <eos>","<sos> "+ f+ " <eos>") for e,f in pairs]
train_pairs = add_tokens(train_pairs)
val_pairs = add_tokens(val_pairs)

In [ ]:
with open("corpus.txt","w", encoding="utf-8") as f:
  for e,fr in train_pairs:
    f.write(e + "\n")
    f.write(fr + "\n")

In [ ]:
spm.SentencePieceTrainer.train(
    input = "corpus.txt",
    model_prefix = "spm",
    vocab_size = 10000,
    character_coverage= 1.0,
    model_type = "bpe"
)

In [ ]:
sp = spm.SentencePieceProcessor()
sp.load("spm.model")

True

#Creating Data Loaders

In [ ]:
def encode_sentences(sentence):
  return sp.encode(sentence, out_type=int)
sos_id = 10000
eos_id = 10001
pad_id = 0
def process_pair(e,f):
  src = [sos_id] + encode_sentences(e) + [eos_id]
  tgt = [sos_id] + encode_sentences(f) + [eos_id]
  return src,tgt
train_data = [process_pair(e,f) for e,f in train_pairs]
val_data = [process_pair(e,f) for e,f in val_pairs]

In [ ]:
def pad_sequence(seq):
  if len(seq) < maxLen:
    return seq +[pad_id] * (maxLen  - len(seq))
  else:
    return seq[:maxLen]
train_data = [(pad_sequence(src),pad_sequence(tgt)) for src,tgt in train_data]
val_data = [(pad_sequence(src),pad_sequence(tgt)) for src,tgt in val_data]

In [ ]:
train_src = torch.tensor([s for s,t in train_data])
train_tgt = torch.tensor([t for s,t in train_data])
val_src = torch.tensor([s for s,t in val_data])
val_tgt = torch.tensor([t for s,t in val_data])

train_dataset = TensorDataset(train_src,train_tgt)
val_dataset = TensorDataset(val_src,val_tgt)

train_loader = DataLoader(train_dataset,batch_size=32,shuffle=True)
val_loader = DataLoader(val_dataset,batch_size=32,shuffle=False)

#Encoder and Decoder

In [ ]:
class Encoder(nn.Module):
    def __init__(self, vocab_size, d_model, max_len, num_layers):
        super().__init__()

        self.token_emb = nn.Embedding(vocab_size, d_model)
        self.pos_embedding = nn.Embedding(max_len, d_model)

        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=8,
            dropout=0.1,
            batch_first=True
        )

        self.transformer_encoder = nn.TransformerEncoder(
            enc_layer,
            num_layers=num_layers
        )

    def forward(self, src):


        padding_mask = (src == 0)

        batch_size, seq_len = src.shape


        positions = torch.arange(0, seq_len).unsqueeze(0).repeat(batch_size, 1).to(src.device)

        src = self.token_emb(src) + self.pos_embedding(positions)


        memory = self.transformer_encoder(
            src,
            src_key_padding_mask=padding_mask
        )

        return memory

In [ ]:
class Decoder(nn.Module):
  def __init__(self,vocab_size,d_model,max_len,num_layers):
    super().__init__()
    self.token_emb = nn.Embedding(vocab_size,d_model)
    self.pos = nn.Embedding(max_len,d_model)
    dec_layer = nn.TransformerDecoderLayer(
        d_model=d_model,
        nhead=8,
        dropout=0.1,
        batch_first=True
    )
    self.transformer_decoder = nn.TransformerDecoder(
        dec_layer,
        num_layers=num_layers
    )
    self.out_proj = nn.Linear(d_model,vocab_size)
  def forward(self,tgt,memory,memory_padding_mask):
    tgt_key_mask = (tgt == 0)
    batch_size, seq_len = tgt.shape

    tgt_mask = torch.triu(
    torch.ones(seq_len, seq_len, device=tgt.device),
    diagonal=1
).bool()

    positions = torch.arange(0, seq_len).unsqueeze(0).repeat(batch_size, 1).to(tgt.device)

    tgt = self.token_emb(tgt) + self.pos(positions)
    decoded = self.transformer_decoder(
        tgt=tgt,
        memory=memory,
        tgt_mask=tgt_mask,
        tgt_key_padding_mask=tgt_key_mask,
        memory_mask = None,
        memory_key_padding_mask=memory_padding_mask
    )
    output = self.out_proj(decoded)
    return output

#Transformer

In [ ]:
class Seq2SeqTransformer(nn.Module):
    def __init__(self, encoder, decoder):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder

    def forward(self, src, tgt):


        memory = self.encoder(src)


        src_padding_mask = (src == 0)


        output = self.decoder(
            tgt,
            memory,
            memory_padding_mask=src_padding_mask
        )

        return output

In [ ]:
VOCAB_SIZE = 10002
D_MODEL = 256
MAX_LEN = 25
NUM_LAYERS = 3
encoder = Encoder(
    vocab_size=VOCAB_SIZE,
    d_model=D_MODEL,
    max_len=MAX_LEN,
    num_layers=NUM_LAYERS
)
decoder = Decoder(
    vocab_size=VOCAB_SIZE,
    d_model=D_MODEL,
    max_len=MAX_LEN,
    num_layers=NUM_LAYERS
)
model = Seq2SeqTransformer(encoder, decoder).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=0.0005)

criterion = nn.CrossEntropyLoss(ignore_index=pad_id)

#Training

#Phase 1

In [ ]:
num_epochs = 10
counter = 0
early_stop = False
best_model = None
best_accuracy = float('-inf')
for epoch in range(num_epochs):
    train_accuracy = 0.0
    val_accuracy = 0.0
    model.train()

    total_tokens = 0
    total_correct = 0
    for src, tgt in train_loader:

        src = src.to(device)
        tgt = tgt.to(device)


        tgt_input = tgt[:, :-1]
        tgt_output = tgt[:, 1:]

        optimizer.zero_grad()

        output = model(src, tgt_input)
        preds = output.argmax(dim=-1)
        mask = (tgt_output != 0)
        correct = (preds == tgt_output) & mask
        total_correct += correct.sum().item()
        total_tokens += mask.sum().item()
        loss = criterion(
            output.reshape(-1, output.shape[-1]),
            tgt_output.reshape(-1)
        )

        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), 1)

        optimizer.step()
    train_accuracy = total_correct / total_tokens
    total_tokens = 0
    total_correct = 0
    model.eval()
    with torch.no_grad():
      for src,tgt in val_loader:
        src = src.to(device)
        tgt = tgt.to(device)

        tgt_input = tgt[:, :-1]
        tgt_output = tgt[:, 1:]
        output = model(src, tgt_input)
        preds = output.argmax(dim=-1)
        mask = (tgt_output != 0)
        correct = (preds == tgt_output) & mask
        total_correct += correct.sum().item()
        total_tokens += mask.sum().item()
      val_accuracy = total_correct / total_tokens
      if val_accuracy > best_accuracy:
        best_accuracy = val_accuracy
        best_model = copy.deepcopy(model)
        counter = 0
      else:
        counter += 1
        if counter == 4:
          print("Early stopping")
          early_stop = True
    if early_stop:
      break
    print(f"Epoch {epoch+1}, Train Acc: {train_accuracy:.2f} , Val acc: {val_accuracy:.2f}")

/usr/local/lib/python3.12/dist-packages/torch/nn/modules/transformer.py:531: UserWarning: The PyTorch API of nested tensors is in prototype stage and will change in the near future. We recommend specifying layout=torch.jagged when constructing a nested tensor, as this layout receives active development, has better operator coverage, and works with torch.compile. (Triggered internally at /pytorch/aten/src/ATen/NestedTensorImpl.cpp:178.)
  output = torch._nested_tensor_from_mask(


Epoch 1, Train Acc: 0.75 , Val acc: 0.79
Epoch 2, Train Acc: 0.82 , Val acc: 0.83
Epoch 3, Train Acc: 0.85 , Val acc: 0.84
Epoch 4, Train Acc: 0.87 , Val acc: 0.85
Epoch 5, Train Acc: 0.89 , Val acc: 0.86
Epoch 6, Train Acc: 0.90 , Val acc: 0.86
Epoch 7, Train Acc: 0.91 , Val acc: 0.87
Epoch 8, Train Acc: 0.91 , Val acc: 0.87
Epoch 9, Train Acc: 0.92 , Val acc: 0.87
Epoch 10, Train Acc: 0.92 , Val acc: 0.87


In [ ]:
model.eval()
with torch.no_grad():
    for src, tgt in val_loader:
        src = src.to(device)
        tgt = tgt.to(device)

        tgt_input = tgt[:, :-1]
        tgt_output = tgt[:, 1:]

        output = model(src, tgt_input)
        preds = output.argmax(dim=-1)

        print("Pred:", preds[0])
        print("True:", tgt_output[0])
        break

Pred: tensor([    4,     6,  9972,   644,   162,    15,    81,   412,  9977,     4,
            5,  9972, 10001,  9977,  9977,  9977,  9977,   162,  9977,    74,
         9977,  9977,  9977,  9977], device='cuda:0')
True: tensor([    4,     6,  9972,   644,   162,    15,    81,   412,    74,     4,
            5,  9972, 10001,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0], device='cuda:0')


#Phase 2

In [ ]:
num_epochs = 10
counter = 0

early_stop = False
best_model = None
best_accuracy = float('-inf')
for epoch in range(num_epochs):
    train_accuracy = 0.0
    val_accuracy = 0.0
    model.train()

    total_tokens = 0
    total_correct = 0
    for src, tgt in train_loader:

        src = src.to(device)
        tgt = tgt.to(device)


        tgt_input = tgt[:, :-1]
        tgt_output = tgt[:, 1:]

        optimizer.zero_grad()

        output = model(src, tgt_input)
        preds = output.argmax(dim=-1)
        mask = (tgt_output != 0)
        correct = (preds == tgt_output) & mask
        total_correct += correct.sum().item()
        total_tokens += mask.sum().item()
        loss = criterion(
            output.reshape(-1, output.shape[-1]),
            tgt_output.reshape(-1)
        )

        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), 1)

        optimizer.step()
    train_accuracy = total_correct / total_tokens
    total_tokens = 0
    total_correct = 0
    model.eval()
    with torch.no_grad():
      for src,tgt in val_loader:
        src = src.to(device)
        tgt = tgt.to(device)

        tgt_input = tgt[:, :-1]
        tgt_output = tgt[:, 1:]
        output = model(src, tgt_input)
        preds = output.argmax(dim=-1)
        mask = (tgt_output != 0)
        correct = (preds == tgt_output) & mask
        total_correct += correct.sum().item()
        total_tokens += mask.sum().item()
      val_accuracy = total_correct / total_tokens
      if val_accuracy > best_accuracy:
        best_accuracy = val_accuracy
        best_model = copy.deepcopy(model)
        counter = 0
      else:
        counter += 1
        if counter == 2:
          print("Early stopping")
          early_stop = True
    if early_stop:
      break
    print(f"Epoch {epoch+1}, Train Acc: {train_accuracy:.2f} , Val acc: {val_accuracy:.2f}")

Epoch 1, Train Acc: 0.93 , Val acc: 0.87
Epoch 2, Train Acc: 0.93 , Val acc: 0.87
Early stopping


#Inference

In [ ]:
test_filtered_pairs = []
for e,f in cleaned_pairs:
  if len(e.split()) <= maxLen and len(f.split()) <= maxLen:
    test_filtered_pairs.append((e,f))
test_pairs = test_filtered_pairs[40000:50000]

In [ ]:
test_pairs = add_tokens(test_pairs)
test_data = [process_pair(e,f) for e,f in test_pairs]
test_data = [(pad_sequence(src),pad_sequence(tgt)) for src,tgt in test_data]

In [ ]:
test_src = torch.tensor([s for s,t in test_data])
test_tgt = torch.tensor([t for s,t in test_data])

test_dataset = TensorDataset(test_src,test_tgt)


testloader = DataLoader(test_dataset,batch_size=32,shuffle=False)


In [ ]:
model.eval()
with torch.no_grad():
  total_correct = 0
  total_tokens = 0
  for src, tgt in testloader:
      src = src.to(device)
      tgt = tgt.to(device)

      tgt_input = tgt[:, :-1]
      tgt_output = tgt[:, 1:]

      output = model(src, tgt_input)
      mask = (tgt_output != 0 )
      preds = output.argmax(dim=-1)
      correct = (preds == tgt_output) & mask
      total_correct += correct.sum().item()
      total_tokens += mask.sum().item()
  inference_accuracy = total_correct / total_tokens
  print(f"Inference accuracy: {inference_accuracy:.2f}")



Inference accuracy: 0.62
